<a href="https://colab.research.google.com/github/Mahendra2409/PyBlender/blob/main/Colab_Script/gcs_to_drive_transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 GCS → Google Drive Transfer

Transfer all rendered images from `gs://pyblender-render-farm/RenderImages/` to `MyDrive/PyBlender/Compare/`

**Auth Strategy:**
- **GCS**: Service account key (`pyblender-e37593034bc1.json`)
- **Drive**: Colab's native Google Drive mount (your Drive Gmail account)

**Drive folder structure:**
```
PyBlender/Compare/
├── boy_01_PC_v2/
│   ├── viridis_colormap/
│   │   ├── boy01.png
│   │   ├── boy01_noisy.png
│   │   └── ...
│   ├── inferno_colormap/
│   └── ... (168 colormap folders)
├── boy_02_pc_v2/
└── ...
```

## Cell 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 2 — Install Dependencies

In [2]:
!pip install -q google-cloud-storage

## Cell 3 — Load GCS Service Account Key

Load the service account JSON key from Colab **Secrets**.
Ensure you have a secret named `GCS_SERVICE_ACCOUNT_KEY` with the content of `pyblender.json`.

In [3]:
from google.colab import userdata
import os

GCS_KEY_PATH = '/tmp/pyblender.json'

if os.path.exists(GCS_KEY_PATH):
    print(f'✓ Key already exists at {GCS_KEY_PATH}')
else:
    try:
        key_content = userdata.get('GCS_SERVICE_ACCOUNT_KEY')
        with open(GCS_KEY_PATH, 'w') as f:
            f.write(key_content)
        print(f'✓ Saved secret to {GCS_KEY_PATH}')
    except userdata.SecretNotFoundError:
        print("✗ Secret 'GCS_SERVICE_ACCOUNT_KEY' not found!")
        print("Please add it to the 'Secrets' tab (key icon) on the left sidebar.")

✓ Saved secret to /tmp/pyblender.json


## Cell 4 — Transfer ALL Files from GCS → Drive

This downloads every file from the bucket and writes it directly to your mounted Google Drive.

- **Resume-safe**: Skips files that already exist on Drive
- **Set `FORCE_OVERWRITE = True`** to re-download everything

In [4]:
import os
import time
from google.cloud import storage
from collections import defaultdict

# ─── Configuration ─────────────────────────────────────────
GCS_KEY_PATH = '/tmp/pyblender.json'
BUCKET_NAME = 'pyblender-render-farm'
GCS_BASE_PATH = 'RenderImages'
DRIVE_BASE_PATH = '/content/drive/MyDrive/PyBlender_Render_Farm/RenderImages'
FORCE_OVERWRITE = False
PROGRESS_INTERVAL = 25
# ───────────────────────────────────────────────────────────

def sizeof_fmt(num_bytes):
    for unit in ['B', 'KB', 'MB', 'GB']:
        if abs(num_bytes) < 1024.0:
            return f'{num_bytes:.1f} {unit}'
        num_bytes /= 1024.0
    return f'{num_bytes:.1f} TB'


def transfer_gcs_to_drive():
    # ── 1. Connect to GCS ────────────────────────────────
    print('🔗 Connecting to GCS...')
    client = storage.Client.from_service_account_json(GCS_KEY_PATH)
    bucket = client.bucket(BUCKET_NAME)

    try:
        next(bucket.list_blobs(max_results=1, prefix=GCS_BASE_PATH + '/'))
        print(f'✓ Connected to bucket: {BUCKET_NAME}')
    except StopIteration:
        print(f'⚠ Bucket is empty or prefix has no files')
        return
    except Exception as e:
        print(f'✗ Failed to access bucket: {e}')
        return

    # ── 2. List ALL blobs ────────────────────────────────
    print(f'\n📋 Listing all files under gs://{BUCKET_NAME}/{GCS_BASE_PATH}/...')
    all_blobs = []
    total_size = 0

    for blob in bucket.list_blobs(prefix=GCS_BASE_PATH + '/'):
        if blob.name.endswith('/'):
            continue
        all_blobs.append(blob)
        total_size += blob.size or 0

    print(f'   Found {len(all_blobs)} files ({sizeof_fmt(total_size)})')

    if not all_blobs:
        print('Nothing to transfer!')
        return

    # ── 3. Analyze structure ─────────────────────────────
    pc_types = defaultdict(lambda: defaultdict(int))
    for blob in all_blobs:
        parts = blob.name[len(GCS_BASE_PATH) + 1:].split('/')
        if len(parts) >= 2:
            pc_types[parts[0]][parts[1]] += 1

    print(f'\n📂 Point Cloud Types found:')
    for pc_type, colormaps in sorted(pc_types.items()):
        total_files = sum(colormaps.values())
        print(f'   ├── {pc_type}: {len(colormaps)} colormaps, {total_files} files')

    # ── 4. Create Drive directory ────────────────────────
    os.makedirs(DRIVE_BASE_PATH, exist_ok=True)
    print(f'\n📁 Drive target: {DRIVE_BASE_PATH}')

    # ── 5. Transfer files ────────────────────────────────
    print(f'\n🚀 Starting transfer...\n')

    transferred = 0
    skipped = 0
    failed = 0
    bytes_transferred = 0
    start_time = time.time()
    failed_files = []

    for i, blob in enumerate(all_blobs):
        relative_path = blob.name[len(GCS_BASE_PATH) + 1:]
        drive_path = os.path.join(DRIVE_BASE_PATH, relative_path)

        # Skip if exists (unless overwrite)
        if not FORCE_OVERWRITE and os.path.exists(drive_path):
            skipped += 1
            continue

        os.makedirs(os.path.dirname(drive_path), exist_ok=True)

        try:
            blob.download_to_filename(drive_path)
            transferred += 1
            bytes_transferred += blob.size or 0
        except Exception as e:
            failed += 1
            failed_files.append((relative_path, str(e)))
            print(f'   ✗ FAILED: {relative_path} — {e}')
            continue

        # Progress
        done = transferred + skipped + failed
        if done % PROGRESS_INTERVAL == 0 or done == len(all_blobs):
            elapsed = time.time() - start_time
            rate = transferred / elapsed if elapsed > 0 else 0
            eta = (len(all_blobs) - done) / rate if rate > 0 else 0
            print(
                f'   [{done}/{len(all_blobs)}] '
                f'✓ {transferred} transferred, ⏭ {skipped} skipped, ✗ {failed} failed '
                f'| {sizeof_fmt(bytes_transferred)} | {rate:.1f} files/s | ETA: {eta:.0f}s'
            )

    # ── 6. Summary ───────────────────────────────────────
    elapsed = time.time() - start_time
    print(f'\n{"="*60}')
    print(f'✅ TRANSFER COMPLETE')
    print(f'{"="*60}')
    print(f'   Transferred : {transferred} files ({sizeof_fmt(bytes_transferred)})')
    print(f'   Skipped     : {skipped} files (already on Drive)')
    print(f'   Failed      : {failed} files')
    print(f'   Total time  : {elapsed:.1f}s ({elapsed/60:.1f} min)')
    if elapsed > 0 and transferred > 0:
        print(f'   Avg speed   : {transferred/elapsed:.1f} files/s')
    print(f'   Drive path  : {DRIVE_BASE_PATH}')
    print(f'{"="*60}')

    if failed_files:
        print(f'\n⚠ Failed files:')
        for path, error in failed_files:
            print(f'   • {path}: {error}')

    return transferred, skipped, failed


# Run the transfer
transfer_gcs_to_drive()

🔗 Connecting to GCS...
⚠ Bucket is empty or prefix has no files


## Cell 5 — Verify Transfer

Check what ended up on Drive

In [5]:
import os
from collections import defaultdict

DRIVE_BASE_PATH = '/content/drive/MyDrive/PyBlender/Compare'

print('🔍 Verifying Drive contents...\n')

stats = defaultdict(lambda: defaultdict(int))
total_files = 0
total_size = 0

for root, dirs, files_list in os.walk(DRIVE_BASE_PATH):
    for f in files_list:
        fpath = os.path.join(root, f)
        rel = os.path.relpath(fpath, DRIVE_BASE_PATH)
        parts = rel.split(os.sep)
        if len(parts) >= 2:
            stats[parts[0]][parts[1]] += 1
        total_files += 1
        total_size += os.path.getsize(fpath)

print(f'📂 {DRIVE_BASE_PATH}')
print(f'   Total: {total_files} files ({total_size / (1024*1024):.1f} MB)\n')

for pc_type in sorted(stats):
    colormaps = stats[pc_type]
    files_count = sum(colormaps.values())
    print(f'   📁 {pc_type}/')
    print(f'      {len(colormaps)} colormap folders, {files_count} files')

print(f'\n✅ Verification complete!')

🔍 Verifying Drive contents...

📂 /content/drive/MyDrive/PyBlender/Compare
   Total: 545 files (736.6 MB)

   📁 boy_01_PC_v2/
      50 colormap folders, 545 files

✅ Verification complete!


In [ ]:
# @title 2.4 Generate Comparison Visualizations (Auto-Detect + Titles)
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 1. Pull settings from your existing config
try:
    from config import CONFIG
    DRIVE_BASE_PATH = CONFIG["DRIVE_BASE_PATH"]
    PC_TYPE = CONFIG["PC_TYPE"]
    COLORMAPS = CONFIG["COLORMAPS"]
except ImportError:
    # Fallback if config isn't in memory
    DRIVE_BASE_PATH = "/content/drive/MyDrive/PyBlender_Render_Farm"
    PC_TYPE = "boy_01_PC_v2"
    COLORMAPS = []

# --- Auto-detect colormaps if the list is empty ---
if not COLORMAPS:
    print("COLORMAPS list is empty. Auto-detecting from Drive...")
    base_render_dir = os.path.join(DRIVE_BASE_PATH, "RenderImages", PC_TYPE)

    if os.path.exists(base_render_dir):
        COLORMAPS = [
            folder.replace("_colormap", "")
            for folder in os.listdir(base_render_dir)
            if os.path.isdir(os.path.join(base_render_dir, folder)) and folder.endswith("_colormap")
        ]
        if COLORMAPS:
            print(f"✅ Successfully detected: {', '.join(COLORMAPS)}\n")
        else:
            print("❌ No colormap directories found. Exiting.")
    else:
        print(f"❌ Base render directory not found: {base_render_dir}")

# 2. Create the target Output Directory
compare_dir = os.path.join(DRIVE_BASE_PATH, "Compare", PC_TYPE)
os.makedirs(compare_dir, exist_ok=True)

# 3. Robust Label & Order Matching
def get_label_and_sort_key(filename):
    name = filename.lower()

    if name.startswith("boy01_noisy"): return "Noisy", 0
    if "bilateral" in name: return "BF", 1
    if "wlop" in name: return "WLOP", 2
    if "ad_" in name: return "AD", 3
    if "dmr" in name: return "DMR", 4
    if "score" in name: return "Score", 5
    if "iterpfn" in name: return "IterPFN", 6
    if "straightpcf" in name: return "StraightPCF", 7
    if "delnoise" in name: return "De(l)Noise", 8
    if "ours" in name: return "Ours", 9
    if name.startswith("boy01."): return "GT", 10

    return filename.split('.')[0].title(), 99

# 4. Generate Visualizations for each Colormap
for colormap in COLORMAPS:
    render_dir = os.path.join(DRIVE_BASE_PATH, "RenderImages", PC_TYPE, f"{colormap}_colormap")

    if not os.path.exists(render_dir):
        continue

    image_files = [f for f in os.listdir(render_dir) if f.endswith('.png') and 'colormap' not in f]

    if not image_files:
        continue

    image_files.sort(key=lambda x: get_label_and_sort_key(x)[1])
    n_images = len(image_files)

    # Setup Figure
    fig, axes = plt.subplots(1, n_images + 1, figsize=(n_images * 3.5, 6), gridspec_kw={'width_ratios': [1]*n_images + [0.15]})
    plt.subplots_adjust(wspace=0.02)

    # --- NEW: Add the Master Title for the Colormap ---
    fig.suptitle(f"Colormap: {colormap.upper()}", fontsize=28, fontweight='bold', fontfamily='serif', y=1.05)

    for i, img_name in enumerate(image_files):
        img_path = os.path.join(render_dir, img_name)
        img = Image.open(img_path)

        # Crop empty background
        bbox = img.getbbox()
        if bbox:
            img = img.crop(bbox)

        axes[i].imshow(img)
        axes[i].axis('off')

        label, _ = get_label_and_sort_key(img_name)
        axes[i].text(0.5, -0.05, label, size=18, ha="center", va="top", transform=axes[i].transAxes, fontfamily='serif')

    # Draw the Vertical Colorbar
    gradient = np.linspace(1, 0, 256).reshape(256, 1)
    axes[-1].imshow(gradient, aspect='auto', cmap=colormap)
    axes[-1].axis('off')

    # Save and Output
    out_path = os.path.join(compare_dir, f"{colormap}_comparison.png")
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0.2, dpi=300, facecolor='white')
    print(f"Saved scaled comparison for {colormap} at: {out_path}")

    # plt.show()

COLORMAPS list is empty. Auto-detecting from Drive...
✅ Successfully detected: Accent, Accent_r, Blues, Blues_r, BrBG, BrBG_r, BuGn, BuGn_r, BuPu, BuPu_r, CMRmap, CMRmap_r, Dark2, Dark2_r, GnBu, GnBu_r, Grays, Grays_r, Greens, Greens_r, Greys, Greys_r, OrRd, OrRd_r, Oranges, Oranges_r, PRGn, PRGn_r, Paired, Paired_r, Pastel1, Pastel1_r, Pastel2, Pastel2_r, PiYG, PiYG_r, PuBuGn, PuBuGn_r, PuBu, PuBu_r, PuOr, PuOr_r, PuRd, PuRd_r, Purples, Purples_r, RdBu, RdBu_r, RdGy, RdGy_r, RdPu, RdPu_r, RdYlBu, RdYlBu_r, RdYlGn, RdYlGn_r, Reds, Reds_r, Set1, Set1_r, Set2, Set2_r, Set3, Set3_r, Spectral, Spectral_r, Wistia, Wistia_r, YlGnBu, YlGnBu_r, YlGn, YlGn_r, YlOrBr, YlOrBr_r, YlOrRd, YlOrRd_r, afmhot, afmhot_r, autumn, autumn_r, berlin, berlin_r, binary, binary_r, bone, bone_r, brg, brg_r, bwr, bwr_r, cividis, cividis_r, cool, cool_r, coolwarm, coolwarm_r, copper, copper_r, cubehelix, cubehelix_r, flag, flag_r, gist_earth, gist_earth_r, gist_gray, gist_gray_r, gist_grey, gist_grey_r, gist_heat

In [ ]:
# @title 2.5 Generate PDF Catalog (Montserrat Headings)
import os
import urllib.request
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image, ImageFile

# Tell Pillow to ignore slightly truncated files caused by Drive syncing
ImageFile.LOAD_TRUNCATED_IMAGES = True

# --- NEW: Fetch and Load Montserrat Font ---
font_url = "https://github.com/JulietaUla/Montserrat/raw/master/fonts/ttf/Montserrat-Bold.ttf"
font_path = "/content/Montserrat-Bold.ttf"

if not os.path.exists(font_path):
    print("Downloading Montserrat font...")
    urllib.request.urlretrieve(font_url, font_path)

# Create a specific FontProperties object to pass to our titles
heading_font = fm.FontProperties(fname=font_path, size=30)

# 1. Pull settings from config
try:
    from config import CONFIG
    DRIVE_BASE_PATH = CONFIG["DRIVE_BASE_PATH"]
    PC_TYPE = CONFIG["PC_TYPE"]
except ImportError:
    # Fallback
    DRIVE_BASE_PATH = "/content/drive/MyDrive/PyBlender_Render_Farm"
    PC_TYPE = "boy_01_PC_v2"

compare_dir = os.path.join(DRIVE_BASE_PATH, "Compare", PC_TYPE)
pdf_path = os.path.join(DRIVE_BASE_PATH, "Compare", f"{PC_TYPE}_Colormap_Catalog.pdf")

# 2. Gather Comparison Images
image_files = [f for f in os.listdir(compare_dir) if f.endswith('_comparison.png')]
image_files.sort()

if not image_files:
    print(f"❌ No comparison images found in {compare_dir}.")
else:
    print(f"Found {len(image_files)} comparison images. Building PDF catalog...")

    # 3. Layout Settings
    IMAGES_PER_PAGE = 6

    # 4. Create multi-page PDF
    with PdfPages(pdf_path) as pdf:
        num_pages = (len(image_files) + IMAGES_PER_PAGE - 1) // IMAGES_PER_PAGE

        for page_idx in range(num_pages):
            fig, axes = plt.subplots(IMAGES_PER_PAGE, 1, figsize=(16, 24))

            if IMAGES_PER_PAGE == 1:
                axes = [axes]

            for i in range(IMAGES_PER_PAGE):
                img_idx = page_idx * IMAGES_PER_PAGE + i
                ax = axes[i]

                if img_idx < len(image_files):
                    img_path = os.path.join(compare_dir, image_files[img_idx])
                    img = Image.open(img_path)
                    ax.imshow(img)

                ax.axis('off')

            # --- APPLIED FONT: Using the 'fontproperties' argument ---
            # title_text = f"{PC_TYPE.replace('_', ' ').title()} - Colormap Catalog (Page {page_idx + 1})"
            # fig.suptitle(title_text, fontproperties=heading_font, color='#222222')

            # plt.tight_layout()
            # plt.subplots_adjust(top=0.92, hspace=0.1)

            pdf.savefig(fig, bbox_inches='tight', facecolor='white')
            plt.close(fig)

    print(f"✅ PDF Catalog successfully saved to: {pdf_path}")

## Cell 6 (Optional) — Cross-Check: Find Missing Files

Compares bucket contents against Drive to ensure **nothing was left behind**

In [ ]:
import os
from google.cloud import storage

GCS_KEY_PATH = '/tmp/pyblender-e37593034bc1.json'
BUCKET_NAME = 'pyblender-render-farm'
GCS_BASE_PATH = 'RenderImages'
DRIVE_BASE_PATH = '/content/drive/MyDrive/PyBlender/Compare'

client = storage.Client.from_service_account_json(GCS_KEY_PATH)
bucket = client.bucket(BUCKET_NAME)

missing = []
matched = 0

for blob in bucket.list_blobs(prefix=GCS_BASE_PATH + '/'):
    if blob.name.endswith('/'):
        continue
    relative_path = blob.name[len(GCS_BASE_PATH) + 1:]
    drive_path = os.path.join(DRIVE_BASE_PATH, relative_path)
    if os.path.exists(drive_path):
        matched += 1
    else:
        missing.append(relative_path)

print(f'✓ Matched on Drive: {matched}')
print(f'✗ Missing from Drive: {len(missing)}')

if missing:
    print(f'\nMissing files:')
    for m in missing[:50]:
        print(f'   • {m}')
    if len(missing) > 50:
        print(f'   ... and {len(missing) - 50} more')
else:
    print(f'\n✅ ALL bucket files are present on Drive! Nothing left behind.')